In [21]:
import matplotlib.pyplot as plt
import numpy as np
import os
import yaml 

from daart.testtube import get_all_params, print_hparams, create_tt_experiment, clean_tt_dir
from test_tube import HyperOptArgumentParser
from daart.data import DataGenerator, compute_sequence_pad, SingleDataset, split_trials
from daart.eval import get_precision_recall, plot_training_curves
from daart.models import Segmenter
from daart.train import Trainer
from daart.transforms import ZScore

from collections import OrderedDict
import logging
import numpy as np
import os
import pandas as pd
import pickle
import torch
from torch.utils import data
from torch.utils.data import SubsetRandomSampler
from typing import List, Union
from typeguard import typechecked

In [33]:
# check to see if the signal is markers/features
# check to see if the "idx" argument of the function is in self.batch_idxs["train"]

class IBLSingleDataset(SingleDataset):
    """Dataset class for a single dataset."""

    @typechecked
    def __getitem__(self, idx: Union[int, np.int64, None]) -> dict:
        """Return batch of data.

        Parameters
        ----------
        idx : int or NoneType
            trial index to load; if `NoneType`, return all data.

        Returns
        -------
        dict
            data sample

        """
        sample = OrderedDict()
        for signal in self.signals:

            # collect signal
            if idx is None:
                sample[signal] = [d for d in self.data[signal]]
            else:
                sample[signal] = [self.data[signal][idx]]

            # transform into tensor
            if not self.as_numpy:
                if self.dtypes[signal] == 'float32':
                    sample[signal] = torch.from_numpy(sample[signal][0]).float()
                else:
                    sample[signal] = torch.from_numpy(sample[signal][0]).long()

        # add batch index
        sample['batch_idx'] = idx

        return sample
    
    
class IBLDataGenerator(DataGenerator):
    """Dataset generator for serving pytorch models.

    This class contains a list of SingleDataset generators. It handles shuffling and iterating
    over these datasets.
    """

    _dtypes = {'train', 'val', 'test'}

    @typechecked
    def __init__(
            self,
            ids_list: List[str],
            signals_list: List[List[str]],
            transforms_list: List[list],
            paths_list: List[List[Union[str, None]]],
            device: str = 'cuda',
            as_numpy: bool = False,
            rng_seed: int = 0,
            trial_splits: Union[str, dict, None] = None,
            train_frac: float = 1.0,
            sequence_length: int = 500,
            batch_size: int = 1,
            num_workers: int = 0,
            pin_memory: bool = False,
            sequence_pad: int = 0,
            input_type: str = 'markers'
    ) -> None:
        """

        Parameters
        ----------
        ids_list : list of strs
            unique identifier for each dataset
        signals_list : list of lists
            list of signals for each dataset
        transforms_list : list of lists
            list of transforms for each dataset
        paths_list : list of lists
            list of paths for each dataset
        device : str, optional
            location of data; options are 'cpu' | 'cuda'
        as_numpy : bool, optional
            if True return data as a numpy array, else return as a torch tensor
        rng_seed : int, optional
            controls split of train/val/test trials
        trial_splits : dict, optional
            determines number of train/val/test trials using the keys 'train_tr', 'val_tr',
            'test_tr', and 'gap_tr'; see :func:`split_trials` for how these are used.
        train_frac : float, optional
            if `0 < train_frac < 1.0`, defines the fraction of assigned training trials to
            actually use; if `train_frac > 1.0`, defines the number of assigned training trials to
            actually use
        sequence_length : int, optional
            number of contiguous data points in a sequence
        batch_size : int, optional
            number of sequences in each batch
        num_workers : int, optional
            number of cpu cores per dataset; defaults to 0 (all data loaded in main process)
        pin_memory : bool, optional
            if True, the data loader automatically pulls fetched data Tensors in pinned memory, and
            thus enables faster transfer to CUDA-enabled GPUs
        sequence_pad : int, optional
            if >0, add `sequence_pad` time points to the beginning/end of each sequence (to account
            for padding with convolution layers)
        input_type : str, optional
            'markers' | 'features'

        """
        self.ids = ids_list
        self.batch_size = batch_size
        self.as_numpy = as_numpy
        self.device = device

        self.datasets = []
        self.signals = signals_list
        self.transforms = transforms_list
        self.paths = paths_list
        for id, signals, transforms, paths in zip(
                ids_list, signals_list, transforms_list, paths_list):
            self.datasets.append(IBLSingleDataset(
                id=id, signals=signals, transforms=transforms, paths=paths, device=device,
                as_numpy=self.as_numpy, sequence_length=sequence_length,
                sequence_pad=sequence_pad, input_type=input_type))

        # collect info about datasets
        self.n_datasets = len(self.datasets)
        self.input_size = self.datasets[0].input_size
        self.feature_names = self.datasets[0].feature_names
        self.label_names = self.datasets[0].label_names

        # get train/val/test batch indices for each dataset
        if trial_splits is None:
            trial_splits = {'train_tr': 8, 'val_tr': 1, 'test_tr': 1, 'gap_tr': 0}
        elif isinstance(trial_splits, str):
            ttypes = ['train_tr', 'val_tr', 'test_tr', 'gap_tr']
            trial_splits = {
                ttype: s for ttype, s in zip(ttypes, [int(s) for s in trial_splits.split(';')])}
        else:
            pass
        self.batch_ratios = [None] * self.n_datasets
        for i, dataset in enumerate(self.datasets):
            dataset.batch_idxs = split_trials(len(dataset), rng_seed=rng_seed, **trial_splits)
            dataset.n_batches = {}
            for dtype in self._dtypes:
                if dtype == 'train':
                    # subsample training data if requested
                    if train_frac != 1.0:
                        n_batches = len(dataset.batch_idxs[dtype])
                        if train_frac < 1.0:
                            # subsample as fraction of total batches
                            n_idxs = int(np.floor(train_frac * n_batches))
                            if n_idxs <= 0:
                                print_str = (
                                    'warning: attempting to use invalid number of training '
                                    'batches; defaulting to all training batches'
                                )
                                logging.info(print_str)
                                n_idxs = n_batches
                        else:
                            # subsample fixed number of batches
                            train_frac = n_batches if train_frac > n_batches else train_frac
                            n_idxs = int(train_frac)
                        idxs_rand = np.random.choice(n_batches, size=n_idxs, replace=False)
                        dataset.batch_idxs[dtype] = dataset.batch_idxs[dtype][idxs_rand]
                    self.batch_ratios[i] = len(dataset.batch_idxs[dtype])
                dataset.n_batches[dtype] = len(dataset.batch_idxs[dtype])
        self.batch_ratios = np.array(self.batch_ratios) / np.sum(self.batch_ratios)

        # find total number of batches per data type; this will be iterated over in the train loop
        # automatically set val/test batch sizes to 1 for more fine-grained logging
        self.n_tot_batches = {}
        for dtype in self._dtypes:
            if dtype == 'train':
                self.n_tot_batches[dtype] = int(np.ceil(np.sum(
                    [dataset.n_batches[dtype] for dataset in self.datasets]) / self.batch_size))
            else:
                self.n_tot_batches[dtype] = np.sum(
                    [dataset.n_batches[dtype] for dataset in self.datasets])

        # create data loaders (will shuffle/batch/etc datasets)
        self.dataset_loaders = [None] * self.n_datasets
        for i, dataset in enumerate(self.datasets):
            self.dataset_loaders[i] = {}
            for dtype in self._dtypes:
                self.dataset_loaders[i][dtype] = torch.utils.data.DataLoader(
                    dataset,
                    batch_size=1,  # keep 1 here so we can combine batches from multiple datasets
                    sampler=SubsetRandomSampler(dataset.batch_idxs[dtype]),
                    num_workers=num_workers,
                    pin_memory=pin_memory)

        # create all iterators (will iterate through data loaders)
        self.dataset_iters = [None] * self.n_datasets
        for i in range(self.n_datasets):
            self.dataset_iters[i] = {}
            for dtype in self._dtypes:
                self.dataset_iters[i][dtype] = iter(self.dataset_loaders[i][dtype])
                
    @typechecked
    def __str__(self) -> str:
        """Pretty printing of dataset info"""
        format_str = str('Generator contains %i IBLSingleDataset objects:\n' % self.n_datasets)
        for dataset in self.datasets:
            format_str += dataset.__str__()
        return format_str


In [34]:
def build_ibl_data_generator(hparams: dict) -> IBLDataGenerator:
    """Helper function to build a data generator from hparam dict."""

    signals = []
    transforms = []
    paths = []

    for expt_id in hparams['expt_ids']:

        signals_curr = []
        transforms_curr = []
        paths_curr = []

        # DLC markers or features (e.g. from simba)
        input_type = hparams.get('input_type', 'markers')
        base_dir = os.path.join(hparams['data_dir'], input_type)
        possible_markers_files = [
            os.path.join(base_dir, expt_id + '_labeled.h5'),
            os.path.join(base_dir, expt_id + '_labeled.csv'),
            os.path.join(base_dir, expt_id + '_labeled.npy'),
            os.path.join(base_dir, expt_id + '.h5'),
            os.path.join(base_dir, expt_id + '.csv'),
            os.path.join(base_dir, expt_id + '.npy'),
        ]
        markers_file = None
        for marker_file_ in possible_markers_files:
            if os.path.exists(marker_file_):
                markers_file = marker_file_
                break
        if markers_file is None:
            msg = f'did not find marker file for {expt_id} in {base_dir}'
            logging.info(msg)
            raise FileNotFoundError(msg)
        signals_curr.append('markers')
        transforms_curr.append(ZScore())
        paths_curr.append(markers_file)

        # hand labels
        if hparams.get('lambda_strong', 0) > 0:
            if expt_id not in hparams.get('expt_ids_to_keep', hparams['expt_ids']):
                hand_labels_file = None
            else:
                base_dir = os.path.join(hparams['data_dir'], 'labels-hand')
                possible_hand_labels_files = [
                    os.path.join(base_dir, expt_id + '_labels.csv'),
                    os.path.join(base_dir, expt_id + '.csv'),
                ]
                hand_labels_file = None
                for hand_labels_file_ in possible_hand_labels_files:
                    if os.path.exists(hand_labels_file_):
                        hand_labels_file = hand_labels_file_
                        break
                if hand_labels_file is None:
                    logging.warning(f'did not find hand labels file for {expt_id} in {base_dir}')
            signals_curr.append('labels_strong')
            transforms_curr.append(None)
            paths_curr.append(hand_labels_file)

        # heuristic labels
        if hparams.get('lambda_weak', 0) > 0:
            base_dir = os.path.join(hparams['data_dir'], 'labels-heuristic')
            possible_heur_labels_files = [
                os.path.join(base_dir, expt_id + '_labels.csv'),
                os.path.join(base_dir, expt_id + '.csv'),
            ]
            heur_labels_file = None
            for heur_labels_file_ in possible_heur_labels_files:
                if os.path.exists(heur_labels_file_):
                    heur_labels_file = heur_labels_file_
                    break
            if heur_labels_file is None:
                logging.warning(f'did not find heuristic labels file for {expt_id} in {base_dir}')
            signals_curr.append('labels_weak')
            transforms_curr.append(None)
            paths_curr.append(heur_labels_file)

        # tasks
        if hparams.get('lambda_task', 0) > 0:
            tasks_labels_file = os.path.join(hparams['data_dir'], 'tasks', expt_id + '.csv')
            signals_curr.append('tasks')
            transforms_curr.append(ZScore())
            paths_curr.append(tasks_labels_file)

        # define data generator signals
        signals.append(signals_curr)
        transforms.append(transforms_curr)
        paths.append(paths_curr)

    # compute padding needed to account for convolutions
    hparams['sequence_pad'] = compute_sequence_pad(hparams)

    # build data generator
    ibl_data_gen = IBLDataGenerator(
        hparams['expt_ids'], signals, transforms, paths,
        device=hparams['device'],
        sequence_length=hparams['sequence_length'],
        sequence_pad=hparams['sequence_pad'],
        batch_size=hparams['batch_size'],
        trial_splits=hparams['trial_splits'],
        train_frac=hparams['train_frac'],
        input_type=hparams.get('input_type', 'markers'),
    )

    # automatically compute input/output sizes from data
    hparams['input_size'] = data_gen.input_size
    hparams['output_size'] = len(data_gen.label_names)

    if hparams.get('lambda_task', 0) > 0:
        task_size = 0
        for batch in data_gen.datasets[0].data['tasks']:
            if batch.shape[1] == 0:
                continue
            else:
                task_size = batch.shape[1]
                break
        hparams['task_size'] = task_size

    return ibl_data_gen

In [35]:
# set config paths
data_config = "/home/bsb2144/daart_utils/configs/data_ibl.yaml"
model_config = "/home/bsb2144/daart_utils/configs/model_ibl.yaml"
train_config = "/home/bsb2144/daart_utils/configs/train_ibl.yaml"

hparams = {}
namespace, extra = parser.parse_known_args()

# add arguments from all configs
configs = [data_config, model_config, train_config]
for config in configs:
    config_dict = yaml.safe_load(open(config))
    for (key, value) in config_dict.items():
        hparams[key] = value


In [36]:
# build data generator
data_gen = build_ibl_data_generator(hparams)
print(data_gen)

Generator contains 5 IBLSingleDataset objects:
danlab_DY_009_2020-02-27-001
    signals: ['markers', 'labels_strong']
    transforms: OrderedDict([('markers', ZScore()), ('labels_strong', None)])
    paths: OrderedDict([('markers', '/home/bsb2144/daart_utils/data/ibl/markers/danlab_DY_009_2020-02-27-001_labeled.npy'), ('labels_strong', '/home/bsb2144/daart_utils/data/ibl/labels-hand/danlab_DY_009_2020-02-27-001_labels.csv')])
danlab_DY_018_2020-10-15-001
    signals: ['markers', 'labels_strong']
    transforms: OrderedDict([('markers', ZScore()), ('labels_strong', None)])
    paths: OrderedDict([('markers', '/home/bsb2144/daart_utils/data/ibl/markers/danlab_DY_018_2020-10-15-001_labeled.npy'), ('labels_strong', '/home/bsb2144/daart_utils/data/ibl/labels-hand/danlab_DY_018_2020-10-15-001_labels.csv')])
hoferlab_SWC_061_2020-11-23-001
    signals: ['markers', 'labels_strong']
    transforms: OrderedDict([('markers', ZScore()), ('labels_strong', None)])
    paths: OrderedDict([('markers',

In [38]:
# see what data generator returns
data, dataset = data_gen.next_batch('train')
print(data.keys())
print()

# batch index per sequence
print(data['batch_idx'])

# shape (n_sequences, sequence_length, n_markers)
print(data['markers'].shape)

# shape (n_sequences, sequence_length)
print(data['labels_strong'].shape)

dict_keys(['markers', 'labels_strong', 'batch_idx'])

tensor([[21],
        [71],
        [63],
        [85],
        [ 6],
        [14],
        [61],
        [27]], device='cuda:0')
torch.Size([8, 1048, 3])
torch.Size([8, 1048])


In [41]:
# build model
model = Segmenter(hparams)
model.to(device='cuda')
print(model)


DTCN architecture
------------------------
Encoder:
    0: DilationBlock
        0: Conv1d(3, 32, kernel_size=(9,), stride=(1,), padding=(4,))
        1: LeakyReLU(negative_slope=0.05)
        2: Dropout2d(p=0.1, inplace=False)
        3: Conv1d(32, 32, kernel_size=(9,), stride=(1,), padding=(4,))
        4: LeakyReLU(negative_slope=0.05)
        5: Dropout2d(p=0.1, inplace=False)
        6: residual connection
        7: LeakyReLU(negative_slope=0.05)

    1: DilationBlock
        0: Conv1d(32, 32, kernel_size=(9,), stride=(1,), padding=(8,), dilation=(2,))
        1: LeakyReLU(negative_slope=0.05)
        2: Dropout2d(p=0.1, inplace=False)
        3: Conv1d(32, 32, kernel_size=(9,), stride=(1,), padding=(8,), dilation=(2,))
        4: LeakyReLU(negative_slope=0.05)
        5: Dropout2d(p=0.1, inplace=False)
        6: residual connection
        7: LeakyReLU(negative_slope=0.05)


Classifier:
    0: Linear(in_features=32, out_features=5, bias=True)




In [42]:
# fit model!
trainer = Trainer(**hparams)
trainer.fit(model, data_gen, save_path=model_save_path)

# save training curves
print('saving training curves to %s' % model_save_path)
plot_training_curves(
    os.path.join(model_save_path, 'metrics.csv'), dtype='train', 
    save_file=os.path.join(model_save_path, 'train_curves'), format='png')
plot_training_curves(
    os.path.join(model_save_path, 'metrics.csv'), dtype='val', 
    save_file=os.path.join(model_save_path, 'val_curves'), format='png')

  0%|          | 0/502 [00:00<?, ?it/s]/home/bsb2144/miniconda3/envs/daart2/lib/python3.7/site-packages/torch/nn/functional.py:1338: UserWarning: dropout2d: Received a 3D input to dropout2d and assuming that channel-wise 1D dropout behavior is desired - input is interpreted as shape (N, C, L), where C is the channel dim. This behavior will change in a future release to interpret the input as one without a batch dimension, i.e. shape (C, H, W). To maintain the 1D channel-wise dropout behavior, please switch to using dropout1d instead.
  warnings.warn("dropout2d: Received a 3D input to dropout2d and assuming that channel-wise "
  0%|          | 0/502 [00:15<?, ?it/s]


KeyboardInterrupt: 